## Introduction

Hello, in this code, I will attempt to find and train the best model between Linear Regression and KNeighbors Regression to see which one can best predict future death ratios. 

## Imports


In [1]:
import kagglehub
# Download latest version
path = kagglehub.dataset_download("iamsouravbanerjee/cause-of-deaths-around-the-world")

print("Path to dataset files:", path)

Path to dataset files: /home/jovyan/.cache/kagglehub/datasets/iamsouravbanerjee/cause-of-deaths-around-the-world/versions/3


In [2]:
# Core packages
import numpy as np
import pandas as pd

# Visualization
import matplotlib.pyplot as plt
import seaborn as sns
from itertools import combinations

# Models
from sklearn.neighbors import KNeighborsRegressor
from sklearn.linear_model import LinearRegression, LogisticRegression
from sklearn.svm import SVC
from sklearn.pipeline import make_pipeline
from sklearn.compose import make_column_transformer
from sklearn.preprocessing import StandardScaler
from sklearn.multioutput import MultiOutputRegressor

# Model selection and evaluation
from sklearn.model_selection import cross_val_score, GridSearchCV, LeaveOneOut, train_test_split
from sklearn.metrics import (accuracy_score, precision_score, recall_score, f1_score, 
                            roc_auc_score, confusion_matrix, classification_report,
                            mean_squared_error, r2_score, mean_absolute_error)
# Preprocessing
from sklearn.preprocessing import OneHotEncoder, StandardScaler, MinMaxScaler
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline, make_pipeline 
from sklearn.model_selection import cross_val_predict  

## Dataset

In [3]:
merged_df = pd.read_csv('merged-df.csv')

In [4]:
# Only take data from before 2019 to prevent skewed results due to COVID-19
merged_df = merged_df[merged_df['Year']<=2019]

In [5]:
disease_cols = [
    "Alzheimer's Disease and Other Dementias",
    "Parkinson's Disease",
    "Cardiovascular Diseases",
    "Neoplasms",
    "Diabetes Mellitus",
    "Chronic Kidney Disease",
    "Chronic Respiratory Diseases",
    "Cirrhosis and Other Chronic Liver Diseases",
    "HIV/AIDS"
]

merged_df[disease_cols] = merged_df[disease_cols].div(merged_df['Total Chronic Deaths'], axis=0)

## Setup

The factors that have been determined to be correlated with the death ratio are GNI per Capita, HDI, Life Expectancy, the Country/Territory, and the Year. Using these factors and data from before 2019, I will attempt to predict the 2019 death ratios of certain countries. The countries that I have picked randomly are all from different regions as to prevent skewed data. The countries are: the United States, Germany, Nigeria, and Thailand.

In [6]:
ct = make_column_transformer(
    (StandardScaler(), ['GNI_per_capita', 'HDI', 'LifeExpectancy', 'Year']),
    (OneHotEncoder(handle_unknown='ignore'), ['Country/Territory']),
    remainder="drop"
)

In [16]:
predict_df = merged_df.copy()
predict_df = predict_df.dropna()

In [26]:
predict_cols = ['GNI_per_capita', 'HDI', 'LifeExpectancy', 'Country/Territory', 'Year']

X_train_US = predict_df[(predict_df['Country/Territory'] == 'United States') & (predict_df['Year'] < 2019)][predict_cols]
y_train_US = predict_df[(predict_df['Country/Territory'] == 'United States') & (predict_df['Year'] < 2019)][disease_cols]
X_test_US = predict_df[(predict_df['Country/Territory'] == 'United States') & (predict_df['Year'] == 2019)][predict_cols]
y_test_US = predict_df[(predict_df['Country/Territory'] == 'United States') & (predict_df['Year'] == 2019)][disease_cols]

X_train_GM = predict_df[(predict_df['Country/Territory'] == 'Germany') & (predict_df['Year'] < 2019)][predict_cols]
y_train_GM = predict_df[(predict_df['Country/Territory'] == 'Germany') & (predict_df['Year'] < 2019)][disease_cols]
X_test_GM = predict_df[(predict_df['Country/Territory'] == 'Germany') & (predict_df['Year'] == 2019)][predict_cols]
y_test_GM = predict_df[(predict_df['Country/Territory'] == 'Germany') & (predict_df['Year'] == 2019)][disease_cols]

X_train_NI = predict_df[(predict_df['Country/Territory'] == 'Nigeria') & (predict_df['Year'] < 2019)][predict_cols]
y_train_NI = predict_df[(predict_df['Country/Territory'] == 'Nigeria') & (predict_df['Year'] < 2019)][disease_cols]
X_test_NI = predict_df[(predict_df['Country/Territory'] == 'Nigeria') & (predict_df['Year'] == 2019)][predict_cols]
y_test_NI = predict_df[(predict_df['Country/Territory'] == 'Nigeria') & (predict_df['Year'] == 2019)][disease_cols]

X_train_TL = predict_df[(predict_df['Country/Territory'] == 'Thailand') & (predict_df['Year'] < 2019)][predict_cols]
y_train_TL = predict_df[(predict_df['Country/Territory'] == 'Thailand') & (predict_df['Year'] < 2019)][disease_cols]
X_test_TL = predict_df[(predict_df['Country/Territory'] == 'Thailand') & (predict_df['Year'] == 2019)][predict_cols]
y_test_TL = predict_df[(predict_df['Country/Territory'] == 'Thailand') & (predict_df['Year'] == 2019)][disease_cols]

## Code

### Linear Regression Model

In [43]:
# Predicting 2019
predict_us = pd.DataFrame([['United States', 2019]], columns = ['Country/Territory', 'Year'])
predict_gm = pd.DataFrame([['Germany', 2019]], columns = ['Country/Territory', 'Year'])
predict_ni = pd.DataFrame([['Nigeria', 2019]], columns = ['Country/Territory', 'Year'])
predict_tl = pd.DataFrame([['Thailand', 2019]], columns = ['Country/Territory', 'Year'])

# United States LR Model
lr_us = make_pipeline(ct, LinearRegression())
multi_lr_us = MultiOutputRegressor(lr_us)
multi_lr_us.fit(X_train_US, y_train_US)
us_pred = multi_lr_us.predict(X_test_US)
mse_us = mean_squared_error(y_test_US, us_pred)

# Germany KNN Model
lr_gm = make_pipeline(ct, LinearRegression())
multi_lr_gm = MultiOutputRegressor(lr_gm)
multi_lr_gm.fit(X_train_GM, y_train_GM)
gm_pred = multi_lr_gm.predict(X_test_GM)
mse_gm = mean_squared_error(y_test_GM, gm_pred)

# Nigeria KNN Model
lr_ni  = make_pipeline(ct, LinearRegression())
multi_lr_ni = MultiOutputRegressor(lr_ni)
multi_lr_ni.fit(X_train_NI, y_train_NI)
ni_pred = multi_lr_ni.predict(X_test_NI)
mse_ni = mean_squared_error(y_test_NI, ni_pred)

# Thailand KNN Model
lr_tl = make_pipeline(ct, LinearRegression())
multi_lr_tl = MultiOutputRegressor(lr_tl)
multi_lr_tl.fit(X_train_TL, y_train_TL)
tl_pred = multi_lr_tl.predict(X_test_TL)
mse_tl = mean_squared_error(y_test_TL, tl_pred)

# Print results
print(f"Linear Regression Model Performance:")
print(f"US Mean Squared Error (MSE): {mse_us:.9f}")
print(f"Germany Mean Squared Error (MSE): {mse_gm:.9f}")
print(f"Nigeria Mean Squared Error (MSE): {mse_ni:.9f}")
print(f"Thailand Mean Squared Error (MSE): {mse_tl:.9f}")

# Cross-validation scores
cv_scores_us = cross_val_score(multi_lr_us, X_train_US, y_train_US, 
                           scoring='neg_mean_squared_error', cv=5)
print(f"US CV MSE: {-cv_scores_us.mean():.9f} (±{-cv_scores_us.std():.9f})")

cv_scores_gm = cross_val_score(multi_lr_gm, X_train_GM, y_train_GM, 
                           scoring='neg_mean_squared_error', cv=5)
print(f"GM CV MSE: {-cv_scores_gm.mean():.9f} (±{-cv_scores_gm.std():.9f})")

cv_scores_ni = cross_val_score(multi_lr_ni, X_train_NI, y_train_NI, 
                           scoring='neg_mean_squared_error', cv=5)
print(f"NI CV MSE: {-cv_scores_ni.mean():.9f} (±{-cv_scores_ni.std():.9f})")

cv_scores_tl = cross_val_score(multi_lr_tl, X_train_TL, y_train_TL, 
                           scoring='neg_mean_squared_error', cv=5)
print(f"TL CV MSE: {-cv_scores_tl.mean():.9f} (±{-cv_scores_tl.std():.9f})")

Linear Regression Model Performance:
US Mean Squared Error (MSE): 0.000011946
Germany Mean Squared Error (MSE): 0.000030758
Nigeria Mean Squared Error (MSE): 0.000002865
Thailand Mean Squared Error (MSE): 0.000026465
US CV MSE: 0.000012190 (±-0.000008790)
GM CV MSE: 0.000041236 (±-0.000042605)
NI CV MSE: 0.000013593 (±-0.000015661)
TL CV MSE: 0.000222526 (±-0.000186996)


### KNeighbors Regression Model

In [44]:
# Compute safe n_neighbors range
def train_knn_model(X_train, y_train, X_test, y_test):
    n_train = len(X_train)
    min_fold_size = n_train - n_train // 5  # Smallest fold size
    max_neighbors = min(100, max(1, min_fold_size - 1))
    n_list = list(range(1, max_neighbors + 1, 10))
    
    param_grid = {
        'columntransformer__standardscaler__with_mean': [True, False],
        'columntransformer__standardscaler__with_std': [True, False],
        'kneighborsregressor__n_neighbors': n_list,
        'kneighborsregressor__metric': ['euclidean', 'manhattan', 'minkowski']
    }
    
    pipeline = make_pipeline(
        ct,
        KNeighborsRegressor()
    )
     
    grid_search = GridSearchCV(pipeline, param_grid, cv=5, scoring='neg_mean_squared_error')
    grid_search.fit(X_train, y_train)
    
    return grid_search.best_estimator_, grid_search

# United States KNN Model
best_us, grid_us = train_knn_model(X_train_US, y_train_US, X_test_US, y_test_US)
us_pred = best_us.predict(X_test_US)
mse_us = mean_squared_error(y_test_US, us_pred)

# Germany KNN Model
best_gm, grid_gm = train_knn_model(X_train_GM, y_train_GM, X_test_GM, y_test_GM)
gm_pred = best_gm.predict(X_test_GM)
mse_gm = mean_squared_error(y_test_GM, gm_pred)

# Nigeria KNN Model
best_ni, grid_ni = train_knn_model(X_train_NI, y_train_NI, X_test_NI, y_test_NI)
ni_pred = best_ni.predict(X_test_NI)
mse_ni = mean_squared_error(y_test_NI, ni_pred)

# Thailand KNN Model
best_tl, grid_tl = train_knn_model(X_train_TL, y_train_TL, X_test_TL, y_test_TL)
tl_pred = best_tl.predict(X_test_TL)
mse_tl = mean_squared_error(y_test_TL, tl_pred)

# Print results
print(f"KNN Model Performance:")
print(f"US Mean Squared Error (MSE): {mse_us:.9f}")
print(f"GM Mean Squared Error (MSE): {mse_gm:.9f}")
print(f"NI Mean Squared Error (MSE): {mse_ni:.9f}")
print(f"TL Mean Squared Error (MSE): {mse_tl:.9f}")

def print_cv_scores(grid, country):
    best_idx = grid.best_index_
    cv_mean = -grid.cv_results_['mean_test_score'][best_idx]
    cv_std = grid.cv_results_['std_test_score'][best_idx]
    print(f"{country} CV MSE: {cv_mean:.9f} (±{cv_std:.9f})")

print("\nCross-Validation Scores:")
print_cv_scores(grid_us, "US")
print_cv_scores(grid_gm, "Germany")
print_cv_scores(grid_ni, "Nigeria")
print_cv_scores(grid_tl, "Thailand")

KNN Model Performance:
US Mean Squared Error (MSE): 0.000000228
GM Mean Squared Error (MSE): 0.000000458
NI Mean Squared Error (MSE): 0.000000553
TL Mean Squared Error (MSE): 0.000003775

Cross-Validation Scores:
US CV MSE: 0.000014011 (±0.000005990)
Germany CV MSE: 0.000052880 (±0.000049984)
Nigeria CV MSE: 0.000012843 (±0.000006756)
Thailand CV MSE: 0.000053758 (±0.000036045)
